# 1. Import of input data

In [ ]:
import nbimporter
import sys
import os
from pathlib import Path

In [ ]:
#Import libraries
try:
    current_dir = RUN_DIR
    print(f"📂 CombinedHeatPower_InternalCombustionEngine.ipynb executed by RUN.ipynb — using directory: {current_dir}")
except NameError:
    current_dir = Path().resolve()
    print(f"📁 CombinedHeatPower_InternalCombustionEngine.ipynb executed standalone — using directory: {current_dir}")

Libraries_path = (current_dir / "Libraries.ipynb").resolve()

# If not found, try in the parent levels
if not Libraries_path.exists():
    Libraries_path = (current_dir.parent / "Libraries.ipynb").resolve()

# Run the file
if Libraries_path.exists():
    print(f"📘 Importing libraries from: {Libraries_path}")
    get_ipython().run_line_magic('run', f'"{Libraries_path}"')
else:
    print("⚠️ Libraries.ipynb not found! Some functions might not be available.")

# 2. Input parameters

In [ ]:
energysystem = pd.read_excel(energysystem_path, sheet_name = 'CHP-Internal Combustion Engine')

In [ ]:
Power = energysystem['Thermal capacity [kW]'];
EfficiencyW = energysystem['Electrical efficiency based on LHV (between 0 and 1)']
EfficiencyQ = energysystem['Heat efficiency based on LHV (between 0 and 1)']
ElectricEnergyproduced = energysystem['Electric energy produced in a year (exported and consumed) [kWh /year]']
ThermalEnergyproduced_primary = energysystem['Thermal energy produced used [kWh /year]']
SupplyTemperature = energysystem['Supply temperature [°C]']
ReturnTemperature = energysystem['Return temperature [°C]']
EnergyInputSourceGasBlend = energysystem['Energy input from gas-blend [kWh/year]']
EnergyInputSourceBiogas = energysystem['Energy input from biogas [kWh/year]']
EnergyInputSourceHydrogen = energysystem['Energy input from hydrogen [kWh/year]']

EquivalentHours = ThermalEnergyproduced_primary / Power

In [ ]:
# DEFAULT PARAMETERS
# Natural gas
LHV = 36.8  # In MJ per Nm3 - Table 3.6 06_V_Natural Gas 3_3 Fuel Data
HHV = 40.4  # In MJ per Nm3 - Table 3.6 06_V_Natural Gas 3_3 Fuel Data

# Biomethane
LHVBiom = 35    # MJ/m3
HHVBiom = 38    # MJ/m3

# Hydrogen
LHVH2 = 120     # MJ/kg
HHVH2 = 142     # MJ/kg 

Lifespan = np.where(
    Power * (EfficiencyW / EfficiencyQ)  < 25,
    40000 / EquivalentHours,
    100000 / EquivalentHours
)

# ADDITIONAL CALCULATIONS
Qused = 1
EfficiencyTot = EfficiencyW + EfficiencyQ
EnergyProducedLHV = (Power * EquivalentHours * RSP) / EfficiencyQ   # fuel = heat / thermal efficiency
EnergyProducedHHV = EnergyProducedLHV * (HHV / LHV)
TotalW = EnergyProducedLHV * EfficiencyW
TotalQ = EnergyProducedLHV * EfficiencyQ * Qused

In [ ]:
# MULTIFUNCTIONALITY AND ALLOCATION OF OUTPUTS
# ------------------------------------------------------------------
RICE_Q_RSP = np.asarray(ThermalEnergyproduced_primary, dtype=float) * RSP          # kWh_th
RICE_W_RSP = RICE_Q_RSP * (np.asarray(EfficiencyW, dtype=float)
                           / np.asarray(EfficiencyQ, dtype=float))                   # kWh_el

# Consistency check against the electricity declared in the input sheet [kWh/year]
_W_input_RSP = np.asarray(ElectricEnergyproduced, dtype=float) * RSP
_mismatch = np.abs(_W_input_RSP - RICE_W_RSP) > 0.05 * np.maximum(RICE_W_RSP, 1e-9)
if np.any(_mismatch & (_W_input_RSP > 0)):
    print("⚠️ RICE: declared electricity differs by >5% from Q·ηW/ηQ; "
          "allocation uses the efficiency-based value.")

# Rows without supply/return temperatures (both 0 °C)
RICE_NoTemperatures = ((np.asarray(SupplyTemperature, dtype=float) == 0)
                       & (np.asarray(ReturnTemperature, dtype=float) == 0))

if Multifunctionality == 'Default':
    # Exergy (Carnot) allocation
    T0_K = float(ExternalTemperature) + 273.15            
    Ts_K = np.asarray(SupplyTemperature, dtype=float) + 273.15
    Tr_K = np.asarray(ReturnTemperature, dtype=float) + 273.15

    with np.errstate(divide='ignore', invalid='ignore'):
        CarnotTemperature = np.where(
            np.isclose(Ts_K, Tr_K),
            Ts_K,                                         # limit of the log-mean for Ts -> Tr
            (Ts_K - Tr_K) / np.log(Ts_K / Tr_K)
        )
        CarnotFactor = 1 - T0_K / CarnotTemperature
        AFQ = (RICE_Q_RSP * CarnotFactor) / (RICE_W_RSP + RICE_Q_RSP * CarnotFactor)

    AFQ_RICE = np.nan_to_num(np.asarray(AFQ, dtype=float).reshape(-1))
    AFW_RICE = 1.0 - AFQ_RICE

else:
    # Power bonus method (EN 15316-4-5): the exported electricity is credited with the
    # primary energy it displaces; the heat takes the remainder of the fuel.
    # Share of the fuel (primary energy) attributed to electricity:
    #   AFW = W * f_el / (F * f_fuel),   AFQ = 1 - AFW
    f_fuel = 1.05   # primary-energy factor of the fuel
    f_el = 2.42     # primary-energy factor of exported electricity
    RICE_F_LHV_RSP = RICE_Q_RSP / np.asarray(EfficiencyQ, dtype=float)   # kWh fuel (LHV)
    AFW_raw = np.asarray((RICE_W_RSP * f_el) / (RICE_F_LHV_RSP * f_fuel), dtype=float).reshape(-1)
    if np.any(AFW_raw > 1):
        print("⚠️ RICE: power bonus exceeds the fuel (AFW > 1); AFW capped at 1, heat burden = 0:",
              np.round(AFW_raw, 3))
    AFW_RICE = np.clip(AFW_raw, 0.0, 1.0)
    AFQ_RICE = 1.0 - AFW_RICE

# No temperatures -> no allocation for that row
AFQ_RICE = np.where(RICE_NoTemperatures, 0.0, AFQ_RICE)
AFW_RICE = np.where(RICE_NoTemperatures, 0.0, AFW_RICE)


# 3. Life cycle inventories

## 3.1 Module A1-3

In [ ]:
# ------------------------------------------------------------------
# SIZE CLASSES (one per engine/row, based on thermal capacity [kW])
#   class: 0 <25 | 1 25-100 | 2 100-300 | 3 300-700 | 4 700-1400 | 5 >=1400
# Class 5 = n modules of the 700-1400 class, n = ceil(Power/1400)
# ------------------------------------------------------------------
_P_RICE = np.asarray(Power, dtype=float)
RICE_SizeClass = np.select(
    [_P_RICE < 25, _P_RICE < 100, _P_RICE < 300, _P_RICE < 700, _P_RICE < 1400],
    [0, 1, 2, 3, 4],
    default=5
)
RICE_nmod = np.where(RICE_SizeClass == 5, np.ceil(_P_RICE / 1400), 1.0)

def _rice_by_class(values):
    """values = 6 numbers, one per size class [0..5]; returns one value per row."""
    return np.asarray(values, dtype=float)[RICE_SizeClass]

# Mass [kg]
mass = _rice_by_class([12783.67, 5133.22, 14070.56, 25247.44, 40903.44, 40903.44])

# Number of component units (ecoinvent unit processes), allocated to heat
RICE_Air        = _rice_by_class([0,   0.4, 1,   1.7, 2.6, 2.6]) * RICE_nmod * AFQ_RICE
RICE_Control    = _rice_by_class([0,   0.4, 1,   1.7, 2.6, 2.6]) * RICE_nmod * AFQ_RICE
RICE_Motor      = _rice_by_class([2.5, 0.3, 1.3, 2.8, 5.3, 5.3]) * RICE_nmod * AFQ_RICE
RICE_Insulation = _rice_by_class([0,   0.4, 1,   1.7, 2.6, 2.6]) * RICE_nmod * AFQ_RICE
RICE_Catalytic  = _rice_by_class([5,   0,   0,   0,   0,   0  ]) * RICE_nmod * AFQ_RICE
RICE_Electric   = _rice_by_class([1,   0.4, 1,   1.7, 2.6, 2.6]) * RICE_nmod * AFQ_RICE
RICE_Generator  = _rice_by_class([1,   0.3, 1.3, 2.8, 5.3, 5.3]) * RICE_nmod * AFQ_RICE
RICE_Sanitary   = _rice_by_class([1,   0.4, 1,   1.7, 2.6, 2.6]) * RICE_nmod * AFQ_RICE
RICE_Storage    = _rice_by_class([2,   0.8, 2,   3.4, 5.2, 5.2]) * RICE_nmod * AFQ_RICE


## 3.2 Module A4

In [ ]:
RICE_Truk_16_32 = ((500 + 200)*(mass)/1000) * AFQ_RICE
RICE_Light_Truk = (50*(mass)/1000) * AFQ_RICE

## 3.3 Module A5

In [ ]:
# Size classes defined in Module A1-3
RICE_energy       = _rice_by_class([0, 0.414, 1, 1, 1.7, 2.61]) * RICE_nmod * AFQ_RICE
RICE_construction = _rice_by_class([0, 0.4,   1, 1, 1.7, 2.61]) * RICE_nmod * AFQ_RICE
RICE_planning     = _rice_by_class([1, 0.414, 1, 1, 1.7, 2.61]) * RICE_nmod * AFQ_RICE
RICE_startup      = _rice_by_class([1, 0.414, 1, 1, 1.7, 2.61]) * RICE_nmod * AFQ_RICE

## 3.4 Module B2

In [ ]:
# Size classes defined in Module A1-3
RICE_maintenance = _rice_by_class([1, 0.377, 1, 1.8, 2.89, 2.89]) * RICE_nmod * AFQ_RICE
RICE_lubricatingoil = ((0.03/1000) * np.asarray(EnergyProducedHHV, dtype=float) * 3.6) * AFQ_RICE

## 3.5 Module B4

In [ ]:
RICENumberReplacements = (np.ceil(np.divide(ReqSL, Lifespan) - 1))
SFRICE = (RSP / ReqSL)  # Scaling factor

## 3.6 Module B6

In [ ]:
#Parameters for repartition of the energy input
GasBlend = EnergyInputSourceGasBlend / (EnergyInputSourceGasBlend + EnergyInputSourceBiogas + EnergyInputSourceHydrogen)
Biogas = EnergyInputSourceBiogas / (EnergyInputSourceGasBlend + EnergyInputSourceBiogas + EnergyInputSourceHydrogen)
Hydrogen = EnergyInputSourceHydrogen / (EnergyInputSourceGasBlend + EnergyInputSourceBiogas + EnergyInputSourceHydrogen)


In [ ]:
# ------------------------------------------------------------------
# B6 - Internal Combustion Engine
# Fuel balances and direct emissions are expressed on an LHV basis.
# ------------------------------------------------------------------

# Useful thermal output over the reference study period [MJ heat]
RICE_heat_output_MJ = (
    np.asarray(ThermalEnergyproduced_primary, dtype=float)
    * RSP
    * 3.6
)

# Total CHP fuel input required for the useful heat [MJ fuel, LHV]
RICE_total_fuel_input_MJ = (
    RICE_heat_output_MJ
    / EfficiencyQ
)

# Fuel input allocated to the thermal co-product [MJ fuel, LHV]
RICE_fuel_energy_MJin = (
    RICE_total_fuel_input_MJ
    * AFQ_RICE
)

# ------------------------------------------------------------------
# Fuel inputs by origin [MJ fuel, LHV]
# ------------------------------------------------------------------

# Gas blend
RICE_GasBlend_NG_MJin = (
    RICE_fuel_energy_MJin
    * GasBlend
    * A
)

RICE_GasBlend_Biomethane_MJin = (
    RICE_fuel_energy_MJin
    * GasBlend
    * B
)

RICE_GasBlend_Hydrogen_MJin = (
    RICE_fuel_energy_MJin
    * GasBlend
    * C
)

# Standalone biomethane
# Legacy variable "Biogas" is treated as biomethane.
RICE_Biomethane_MJin = (
    RICE_fuel_energy_MJin
    * Biogas
)

# Standalone hydrogen
RICE_Hydrogen_MJin = (
    RICE_fuel_energy_MJin
    * Hydrogen
)

# ------------------------------------------------------------------
# Inputs to ecoinvent
# ------------------------------------------------------------------

# Natural gas [m3]
RICE_GasBlend_NG_Input = (
    RICE_GasBlend_NG_MJin
    / LHV
)

# Biomethane [m3]
RICE_GasBlend_Biomethane_Input = (
    RICE_GasBlend_Biomethane_MJin
    / LHVBiom
)

RICE_Biomethane_Input = (
    RICE_Biomethane_MJin
    / LHVBiom
)

# Hydrogen:
# MJ_H2 -> kWh_H2 -> kg_H2 -> kWh electricity from photovoltaic system
RICE_LHVH2_kWh_kg = (
    LHVH2
    / 3.6
)

RICE_GasBlend_Hydrogen_kWhH2 = (
    RICE_GasBlend_Hydrogen_MJin
    / 3.6
)

RICE_GasBlend_Hydrogen_kg = (
    RICE_GasBlend_Hydrogen_kWhH2
    / RICE_LHVH2_kWh_kg
)

RICE_GasBlend_Hydrogen_Input = (
    RICE_GasBlend_Hydrogen_kg
    * 55
)

RICE_Hydrogen_kWhH2 = (
    RICE_Hydrogen_MJin
    / 3.6
)

RICE_Hydrogen_kg = (
    RICE_Hydrogen_kWhH2
    / RICE_LHVH2_kWh_kg
)

RICE_Hydrogen_Input = (
    RICE_Hydrogen_kg
    * 55
)

# ------------------------------------------------------------------
# Fuel-energy aggregates for direct emissions [MJin, LHV]
# ------------------------------------------------------------------

RICE_non_hydrogen_energy_MJin = (
    RICE_GasBlend_NG_MJin
    + RICE_GasBlend_Biomethane_MJin
    + RICE_Biomethane_MJin
)

RICE_fossil_energy_MJin = (
    RICE_GasBlend_NG_MJin
)

RICE_biogenic_energy_MJin = (
    RICE_GasBlend_Biomethane_MJin
    + RICE_Biomethane_MJin
)

# Available for future H2-specific direct-emission factors
RICE_hydrogen_energy_MJin = (
    RICE_GasBlend_Hydrogen_MJin
    + RICE_Hydrogen_MJin
)

# ------------------------------------------------------------------
# Direct emissions
# Factors are per MJ fuel input, LHV basis.
# ------------------------------------------------------------------

RICE_Acetaldehyde = 0.0000000008 * RICE_non_hydrogen_energy_MJin
RICE_AceticAcid = 0.000000121 * RICE_non_hydrogen_energy_MJin
RICE_Benzene = 0.000000000926 * RICE_non_hydrogen_energy_MJin
RICE_Benzoapyrene = 0.000000000000529 * RICE_non_hydrogen_energy_MJin
RICE_Butane = 0.000000926 * RICE_non_hydrogen_energy_MJin

RICE_CarbonDioxideFossil = (
    0.0561
    * RICE_fossil_energy_MJin
)

RICE_CarbonDioxideBiogenic = (
    0.0561
    * RICE_biogenic_energy_MJin
)

RICE_CarbonMonoxideFossil = (
    0.000003
    * RICE_non_hydrogen_energy_MJin
)

RICE_DinitrogenMonoxide = 0.000001 * RICE_non_hydrogen_energy_MJin
RICE_Ethane = 0.00000137 * RICE_non_hydrogen_energy_MJin
RICE_Formaldehyde = 0.0000000331 * RICE_non_hydrogen_energy_MJin
RICE_Hexane = 0.000000793 * RICE_non_hydrogen_energy_MJin
RICE_Mercury = 0.00000000003 * RICE_non_hydrogen_energy_MJin
RICE_MethaneFossil = 0.000001 * RICE_non_hydrogen_energy_MJin
RICE_NitrogenOxides = 0.000064 * RICE_non_hydrogen_energy_MJin
RICE_PAH = 0.000000008 * RICE_non_hydrogen_energy_MJin
RICE_Particulates = 0.0000005 * RICE_non_hydrogen_energy_MJin
RICE_Pentane = 0.00000115 * RICE_non_hydrogen_energy_MJin
RICE_Propane = 0.000000705 * RICE_non_hydrogen_energy_MJin
RICE_PropionicAcid = 0.000000016 * RICE_non_hydrogen_energy_MJin
RICE_SulfurDioxide = 0.0000005 * RICE_non_hydrogen_energy_MJin
RICE_Toluene = 0.0000000015 * RICE_non_hydrogen_energy_MJin
RICE_Dioxins = 0.000000000029 * RICE_non_hydrogen_energy_MJin

## 3.7 Module C2

In [ ]:
RICE_Transport_to_treatment_plant = 50 * (mass/1000) * AFQ_RICE

## 3.8 Module C3

In [ ]:
# Material content [kg] = kg per component unit x number of units from Module A1-3
RICE_reinforcingsteelAirinput        = 288 * RICE_Air

RICE_aluminiumControlcabinet         = 0.15 * RICE_Control
RICE_copperControlcabinet            = 10.8 * RICE_Control
RICE_leadControlcabinet              = 0.76 * RICE_Control
RICE_nickelControlcabinet            = 0.34 * RICE_Control
RICE_platinumControlcabinet          = 0.0021 * RICE_Control
RICE_HDPEControlcabinet              = 78.5 * RICE_Control
RICE_PVCemulsionatedControlcabinet   = 0.95745 * RICE_Control
RICE_PVCpolimerizedControlcabinet    = 6.54255 * RICE_Control
RICE_reinforcingsteelControlcabinet  = 276 * RICE_Control
RICE_tinControlcabinet               = 1.62 * RICE_Control
RICE_zincControlcabinet              = 0.25 * RICE_Control

RICE_castironGasmotor                = 1000 * RICE_Motor
RICE_reinforcingsteelGasmotor        = 100 * RICE_Motor
RICE_chromiumsteelGasmotor           = 100 * RICE_Motor
RICE_lowalloyedsteelGasmotor         = 200 * RICE_Motor

RICE_reinforcingsteelSoundinsulation = 1920 * RICE_Insulation
RICE_stonewoolSoundinsulation        = 480 * RICE_Insulation

RICE_corrugatedboxboardConverter     = 0.55 * RICE_Catalytic
RICE_palladiumConverter              = 0.0082 * RICE_Catalytic
RICE_platinumConverter               = 0.0409 * RICE_Catalytic
RICE_rhodiumConverter                = 0.0082 * RICE_Catalytic
RICE_chromiumsteelConverter          = 66 * RICE_Catalytic
RICE_zeoliteConverter                = 3.8 * RICE_Catalytic

RICE_aluminiumEle                    = 0.1 * RICE_Electric
RICE_copperEle                       = 6.5 * RICE_Electric
RICE_leadEle                         = 0.47 * RICE_Electric
RICE_nickelEle                       = 0.21 * RICE_Electric
RICE_platinumEle                     = 0.0013 * RICE_Electric
RICE_HDPEEle                         = 47.4 * RICE_Electric
RICE_PVCpolimerizedEle               = 0.501133 * RICE_Electric
RICE_reinforcingsteelEle             = 52.7 * RICE_Electric
RICE_tinEle                          = 0.997 * RICE_Electric
RICE_zincEle                         = 0.154 * RICE_Electric

RICE_castironGen                     = 595 * RICE_Generator
RICE_copperGen                       = 255 * RICE_Generator

RICE_reinforcingsteelSan             = 353 * RICE_Sanitary
RICE_stonewoolSan                    = 7 * RICE_Sanitary

RICE_alkydpaintStorage               = 4.2 * RICE_Storage
RICE_aluminiumStorage                = 87.2 * RICE_Storage
RICE_reinforcingsteelStorage         = 1770 * RICE_Storage
RICE_sheetrollingaluminiumStorage    = 87.2 * RICE_Storage
RICE_sheetrollingsteelStorage        = 1770 * RICE_Storage
RICE_chromiumsteelStorage            = 3.7 * RICE_Storage
RICE_stonewoolStorage                = 84 * RICE_Storage

## 3.9 Module C4

In [ ]:
RICE_EOL_landfill_reinforcingsteelAirinput = RICE_reinforcingsteelAirinput * LandfillMetals
RICE_EOL_landfill_aluminiumControlcabinet = RICE_aluminiumControlcabinet * LandfillMetals
RICE_EOL_landfill_copperControlcabinet = RICE_copperControlcabinet * LandfillMetals
RICE_EOL_landfill_leadControlcabinet = RICE_leadControlcabinet * LandfillMetals2
RICE_EOL_landfill_nickelControlcabinet = RICE_nickelControlcabinet * LandfillMetals2
RICE_EOL_landfill_platinumControlcabinet = RICE_platinumControlcabinet * LandfillMetals2
RICE_EOL_inc_HDPEControlcabinet = RICE_HDPEControlcabinet * IncinerationPEHD
RICE_EOL_landfill_HDPEControlcabinet = RICE_HDPEControlcabinet * LandfillPEHD
RICE_EOL_inc_PVCemulsionatedControlcabinet = RICE_PVCemulsionatedControlcabinet * IncinerationPVC
RICE_EOL_landfill_PVCemulsionatedControlcabinet = RICE_PVCemulsionatedControlcabinet * LandfillPVC
RICE_EOL_inc_PVCpolimerizedControlcabinet = RICE_PVCpolimerizedControlcabinet * IncinerationPVC
RICE_EOL_landfill_PVCpolimerizedControlcabinet = RICE_PVCpolimerizedControlcabinet * LandfillPVC
RICE_EOL_landfill_reinforcingsteelControlcabinet = RICE_reinforcingsteelControlcabinet * LandfillMetals
RICE_EOL_landfill_tinControlcabinet = RICE_tinControlcabinet * LandfillMetals
RICE_EOL_landfill_zincControlcabinet = RICE_zincControlcabinet * LandfillMetals2
RICE_EOL_landfill_castironGasmotor = RICE_castironGasmotor * LandfillMetals2
RICE_EOL_landfill_reinforcingsteelGasmotor = RICE_reinforcingsteelGasmotor * LandfillMetals
RICE_EOL_landfill_chromiumsteelGasmotor = RICE_chromiumsteelGasmotor * LandfillMetals2
RICE_EOL_landfill_lowalloyedsteelGasmotor = RICE_lowalloyedsteelGasmotor * LandfillMetals
RICE_EOL_landfill_reinforcingsteelSoundinsulation = RICE_reinforcingsteelSoundinsulation * LandfillMetals
RICE_EOL_landfill_stonewoolSoundinsulation = RICE_stonewoolSoundinsulation * LandfillMineralWool
RICE_EOL_inc_corrugatedboxboardConverter = RICE_corrugatedboxboardConverter * IncinerationCardboard
RICE_EOL_landfill_corrugatedboxboardConverter = RICE_corrugatedboxboardConverter * LandfillCardboard
RICE_EOL_landfill_palladiumConverter = RICE_palladiumConverter * LandfillMetals2
RICE_EOL_landfill_platinumConverter = RICE_platinumConverter * LandfillMetals2
RICE_EOL_landfill_rhodiumConverter = RICE_rhodiumConverter * LandfillMetals2
RICE_EOL_landfill_chromiumsteelConverter = RICE_chromiumsteelConverter * LandfillMetals2
RICE_EOL_inc_zeoliteConverter = RICE_zeoliteConverter
RICE_EOL_landfill_aluminiumEle = RICE_aluminiumEle * LandfillMetals
RICE_EOL_landfill_copperEle = RICE_copperEle * LandfillMetals
RICE_EOL_landfill_leadEle = RICE_leadEle * LandfillMetals2
RICE_EOL_landfill_nickelEle = RICE_nickelEle * LandfillMetals2
RICE_EOL_landfill_platinumEle = RICE_platinumEle * LandfillMetals2
RICE_EOL_inc_HDPEEle = RICE_HDPEEle * IncinerationPEHD
RICE_EOL_landfill_HDPEEle = RICE_HDPEEle * LandfillPEHD
RICE_EOL_inc_PVCpolimerizedEle = RICE_PVCpolimerizedEle * IncinerationPVC
RICE_EOL_landfill_PVCpolimerizedEle = RICE_PVCpolimerizedEle * LandfillPVC
RICE_EOL_landfill_reinforcingsteelEle = RICE_reinforcingsteelEle * LandfillMetals
RICE_EOL_landfill_tinEle = RICE_tinEle * LandfillMetals
RICE_EOL_landfill_zincEle = RICE_zincEle * LandfillMetals2
RICE_EOL_landfill_castironGen = RICE_castironGen * LandfillMetals2
RICE_EOL_landfill_copperGen = RICE_copperGen * LandfillMetals
RICE_EOL_landfill_reinforcingsteelSan = RICE_reinforcingsteelSan * LandfillMetals
RICE_EOL_landfill_stonewoolSan = RICE_stonewoolSan * LandfillMineralWool
RICE_EOL_inc_alkydpaintStorage = RICE_alkydpaintStorage
RICE_EOL_landfill_aluminiumStorage = RICE_aluminiumStorage * LandfillMetals
RICE_EOL_landfill_reinforcingsteelStorage = RICE_reinforcingsteelStorage * LandfillMetals
RICE_EOL_landfill_sheetrollingaluminiumStorage = RICE_sheetrollingaluminiumStorage * LandfillMetals
RICE_EOL_landfill_sheetrollingsteelStorage = RICE_sheetrollingsteelStorage * LandfillMetals
RICE_EOL_landfill_chromiumsteelStorage = RICE_chromiumsteelStorage * LandfillMetals2
RICE_EOL_landfill_stonewoolStorage = RICE_stonewoolStorage * LandfillMineralWool

## 3.10 Module D1 (related to the export of secondary materials)

In [ ]:
RICE_recycling_reinforcingsteelAirinput = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelAirinput) - (RecycledContentMetals * RICE_reinforcingsteelAirinput)
RICE_primary_reinforcingsteelAirinput = ((RecycledContentMetals * RICE_reinforcingsteelAirinput) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelAirinput)) * SRmetals
RICE_recycling_aluminiumControlcabinet = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_aluminiumControlcabinet) - (RecycledContentMetals * RICE_aluminiumControlcabinet)
RICE_primary_aluminiumControlcabinet = ((RecycledContentMetals * RICE_aluminiumControlcabinet) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_aluminiumControlcabinet)) * SRmetals
RICE_recycling_copperControlcabinet = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_copperControlcabinet) - (RecycledContentMetals * RICE_copperControlcabinet)
RICE_primary_copperControlcabinet = ((RecycledContentMetals * RICE_copperControlcabinet) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_copperControlcabinet)) * SRmetals
RICE_recycling_leadControlcabinet = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_leadControlcabinet) - (RecycledContentMetals * RICE_leadControlcabinet)
RICE_primary_leadControlcabinet = ((RecycledContentMetals * RICE_leadControlcabinet) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_leadControlcabinet)) * SRmetals
RICE_recycling_nickelControlcabinet = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_nickelControlcabinet) - (RecycledContentMetals * RICE_nickelControlcabinet)
RICE_primary_nickelControlcabinet = ((RecycledContentMetals * RICE_nickelControlcabinet) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_nickelControlcabinet)) * SRmetals
RICE_recycling_platinumControlcabinet = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_platinumControlcabinet) - (RecycledContentMetals * RICE_platinumControlcabinet)
RICE_primary_platinumControlcabinet = ((RecycledContentMetals * RICE_platinumControlcabinet) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_platinumControlcabinet)) * SRmetals
RICE_recycling_HDPEControlcabinet = (RecyclingPEHD * RecyclingEfficiencyPlastics * RICE_tinControlcabinet) - (RecycledContentPlastic * RICE_tinControlcabinet)
RICE_primary_HDPEControlcabinet = ((RecycledContentPlastic * RICE_tinControlcabinet) - (RecyclingPEHD * RecyclingEfficiencyPlastics * RICE_tinControlcabinet)) * SRmetals
RICE_recycling_reinforcingsteelControlcabinet = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelControlcabinet) - (RecycledContentMetals * RICE_reinforcingsteelControlcabinet)
RICE_primary_reinforcingsteelControlcabinet = ((RecycledContentMetals * RICE_reinforcingsteelControlcabinet) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelControlcabinet)) * SRmetals
RICE_recycling_tinControlcabinet = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_tinControlcabinet) - (RecycledContentMetals * RICE_tinControlcabinet)
RICE_primary_tinControlcabinet = ((RecycledContentMetals * RICE_tinControlcabinet) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_tinControlcabinet)) * SRmetals
RICE_recycling_zincControlcabinet = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_zincControlcabinet) - (RecycledContentMetals * RICE_zincControlcabinet)
RICE_primary_zincControlcabinet = ((RecycledContentMetals * RICE_zincControlcabinet) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_zincControlcabinet)) * SRmetals
RICE_recycling_castironGasmotor = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_castironGasmotor) - (RecycledContentMetals * RICE_castironGasmotor)
RICE_primary_castironGasmotor = ((RecycledContentMetals * RICE_castironGasmotor) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_castironGasmotor)) * SRmetals
RICE_recycling_reinforcingsteelGasmotor = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelGasmotor) - (RecycledContentMetals * RICE_reinforcingsteelGasmotor)
RICE_primary_reinforcingsteelGasmotor = ((RecycledContentMetals * RICE_reinforcingsteelGasmotor) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelGasmotor)) * SRmetals
RICE_recycling_chromiumsteelGasmotor = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_chromiumsteelGasmotor) - (RecycledContentMetals * RICE_chromiumsteelGasmotor)
RICE_primary_chromiumsteelGasmotor = ((RecycledContentMetals * RICE_chromiumsteelGasmotor) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_chromiumsteelGasmotor)) * SRmetals
RICE_recycling_lowalloyedsteelGasmotor = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_lowalloyedsteelGasmotor) - (RecycledContentMetals * RICE_lowalloyedsteelGasmotor)
RICE_primary_lowalloyedsteelGasmotor = ((RecycledContentMetals * RICE_lowalloyedsteelGasmotor) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_lowalloyedsteelGasmotor)) * SRmetals
RICE_recycling_reinforcingsteelSoundinsulation = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelSoundinsulation) - (RecycledContentMetals * RICE_reinforcingsteelSoundinsulation)
RICE_primary_reinforcingsteelSoundinsulation = ((RecycledContentMetals * RICE_reinforcingsteelSoundinsulation) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelSoundinsulation)) * SRmetals
RICE_recycling_corrugatedboxboardConverter = (RecyclingCardboard * RecyclingEfficiencyCardboard * RICE_corrugatedboxboardConverter) - (RecycledContentCardboard * RICE_corrugatedboxboardConverter)
RICE_primary_corrugatedboxboardConverter = ((RecycledContentCardboard * RICE_corrugatedboxboardConverter) - (RecyclingCardboard * RecyclingEfficiencyCardboard * RICE_corrugatedboxboardConverter)) * SRinert
RICE_recycling_palladiumConverter = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_palladiumConverter) - (RecycledContentMetals * RICE_palladiumConverter)
RICE_primary_palladiumConverter = ((RecycledContentMetals * RICE_palladiumConverter) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_palladiumConverter)) * SRmetals
RICE_recycling_platinumConverter = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_platinumConverter) - (RecycledContentMetals * RICE_platinumConverter)
RICE_primary_platinumConverter = ((RecycledContentMetals * RICE_platinumConverter) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_platinumConverter)) * SRmetals
RICE_recycling_rhodiumConverter = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_rhodiumConverter) - (RecycledContentMetals * RICE_rhodiumConverter)
RICE_primary_rhodiumConverter = ((RecycledContentMetals * RICE_rhodiumConverter) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_rhodiumConverter)) * SRmetals
RICE_recycling_chromiumsteelConverter = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_chromiumsteelConverter) - (RecycledContentMetals * RICE_chromiumsteelConverter)
RICE_primary_chromiumsteelConverter = ((RecycledContentMetals * RICE_chromiumsteelConverter) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_chromiumsteelConverter)) * SRmetals
RICE_recycling_aluminiumEle = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_aluminiumEle) - (RecycledContentMetals * RICE_aluminiumEle)
RICE_primary_aluminiumEle = ((RecycledContentMetals * RICE_aluminiumEle) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_aluminiumEle)) * SRmetals
RICE_recycling_copperEle = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_copperEle) - (RecycledContentMetals * RICE_copperEle)
RICE_primary_copperEle = ((RecycledContentMetals * RICE_copperEle) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_copperEle)) * SRmetals
RICE_recycling_leadEle = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_leadEle) - (RecycledContentMetals * RICE_leadEle)
RICE_primary_leadEle = ((RecycledContentMetals * RICE_leadEle) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_leadEle)) * SRmetals
RICE_recycling_nickelEle = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_nickelEle) - (RecycledContentMetals * RICE_nickelEle)
RICE_primary_nickelEle = ((RecycledContentMetals * RICE_nickelEle) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_nickelEle)) * SRmetals
RICE_recycling_platinumEle = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_platinumEle) - (RecycledContentMetals * RICE_platinumEle)
RICE_primary_platinumEle = ((RecycledContentMetals * RICE_platinumEle) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_platinumEle)) * SRmetals
RICE_recycling_HDPEEle = (RecyclingPEHD * RecyclingEfficiencyPlastics * RICE_HDPEEle) - (RecycledContentPlastic * RICE_HDPEEle)
RICE_primary_HDPEEle = ((RecycledContentPlastic * RICE_HDPEEle) - (RecyclingPEHD * RecyclingEfficiencyPlastics * RICE_HDPEEle)) * SRinert
RICE_recycling_reinforcingsteelEle = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelEle) - (RecycledContentMetals * RICE_reinforcingsteelEle)
RICE_primary_reinforcingsteelEle = ((RecycledContentMetals * RICE_reinforcingsteelEle) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelEle)) * SRmetals
RICE_recycling_reinforcingsteelEle = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelEle) - (RecycledContentMetals * RICE_reinforcingsteelEle)
RICE_primary_reinforcingsteelEle = ((RecycledContentMetals * RICE_reinforcingsteelEle) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelEle)) * SRmetals
RICE_recycling_zincEle = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_zincEle) - (RecycledContentMetals * RICE_zincEle)
RICE_primary_zincEle = ((RecycledContentMetals * RICE_zincEle) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_zincEle)) * SRmetals
RICE_recycling_castironGen = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_castironGen) - (RecycledContentMetals * RICE_castironGen)
RICE_primary_castironGen = ((RecycledContentMetals * RICE_castironGen) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_castironGen)) * SRmetals
RICE_recycling_copperGen = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_copperGen) - (RecycledContentMetals * RICE_copperGen)
RICE_primary_copperGen = ((RecycledContentMetals * RICE_copperGen) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_copperGen)) * SRmetals
RICE_recycling_reinforcingsteelSan = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelSan) - (RecycledContentMetals * RICE_reinforcingsteelSan)
RICE_primary_reinforcingsteelSan = ((RecycledContentMetals * RICE_reinforcingsteelSan) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelSan)) * SRmetals
RICE_recycling_aluminiumStorage = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_aluminiumStorage) - (RecycledContentMetals * RICE_aluminiumStorage)
RICE_primary_aluminiumStorage = ((RecycledContentMetals * RICE_aluminiumStorage) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_aluminiumStorage)) * SRmetals
RICE_recycling_reinforcingsteelStorage = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelStorage) - (RecycledContentMetals * RICE_reinforcingsteelStorage)
RICE_primary_reinforcingsteelStorage = ((RecycledContentMetals * RICE_reinforcingsteelStorage) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_reinforcingsteelStorage)) * SRmetals
RICE_recycling_sheetrollingaluminiumStorage = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_sheetrollingaluminiumStorage) - (RecycledContentMetals * RICE_sheetrollingaluminiumStorage)
RICE_primary_sheetrollingaluminiumStorage = ((RecycledContentMetals * RICE_sheetrollingaluminiumStorage) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_sheetrollingaluminiumStorage)) * SRmetals
RICE_recycling_sheetrollingsteelStorage = (RecyclingMetals * RecyclingEfficiencyMetals * RICE_sheetrollingsteelStorage) - (RecycledContentMetals * RICE_sheetrollingsteelStorage)
RICE_primary_sheetrollingsteelStorage = ((RecycledContentMetals * RICE_sheetrollingsteelStorage) - (RecyclingMetals * RecyclingEfficiencyMetals * RICE_sheetrollingsteelStorage)) * SRmetals
RICE_recycling_chromiumsteelStorage = (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_chromiumsteelStorage) - (RecycledContentMetals * RICE_chromiumsteelStorage)
RICE_primary_chromiumsteelStorage = ((RecycledContentMetals * RICE_chromiumsteelStorage) - (RecyclingMetals2 * RecyclingEfficiencyMetals * RICE_chromiumsteelStorage)) * SRmetals

## 3.11 Module D3 (related to the export of energy as result of waste incineration)

In [ ]:
RICE_energyrecoveryheat_HDPEControlcabinet =  - RICE_EOL_inc_HDPEControlcabinet * LHVHDPE * Xincheat
RICE_energyrecoveryele_HDPEControlcabinet =  (- RICE_EOL_inc_HDPEControlcabinet * LHVHDPE * Xincele) / 3.6

RICE_energyrecoveryheat_PVCemulsionatedControlcabinet =  - RICE_EOL_inc_PVCemulsionatedControlcabinet * LHVPVC * Xincheat
RICE_energyrecoveryele_PVCemulsionatedControlcabinet =  (- RICE_EOL_inc_PVCemulsionatedControlcabinet * LHVPVC * Xincele) / 3.6

RICE_energyrecoveryheat_PVCpolimerizedControlcabinet =  - RICE_EOL_inc_PVCpolimerizedControlcabinet * LHVPVC * Xincheat
RICE_energyrecoveryele_PVCpolimerizedControlcabinet =  (- RICE_EOL_inc_PVCpolimerizedControlcabinet * LHVPVC * Xincele) / 3.6

RICE_energyrecoveryheat_corrugatedboxboardConverter =  - RICE_EOL_inc_corrugatedboxboardConverter * LHVCardboard * Xincheat
RICE_energyrecoveryele_corrugatedboxboardConverter =  (- RICE_EOL_inc_corrugatedboxboardConverter * LHVCardboard * Xincele) / 3.6

RICE_energyrecoveryheat_HDPEEle =  - RICE_EOL_inc_HDPEEle * LHVHDPE * Xincheat
RICE_energyrecoveryele_HDPEEle =  (- RICE_EOL_inc_HDPEEle * LHVHDPE * Xincele) / 3.6

RICE_energyrecoveryheat_PVCpolimerizedEle =  - RICE_EOL_inc_PVCpolimerizedEle * LHVPVC * Xincheat
RICE_energyrecoveryele_PVCpolimerizedEle =  (- RICE_EOL_inc_PVCpolimerizedEle * LHVPVC * Xincele) / 3.6

RICE_energyrecoveryheat_alkydpaintStorage = - RICE_EOL_inc_alkydpaintStorage * LHVHazardous * Xincheat
RICE_energyrecoveryele_alkydpaintStorage = (- RICE_EOL_inc_alkydpaintStorage * LHVHazardous * Xincele) / 3.6

RICE_energyrecoveryheat_zeoliteConverter = - RICE_EOL_inc_zeoliteConverter * LHVHazardous * Xincheat
RICE_energyrecoveryele_zeoliteConverter = (- RICE_EOL_inc_zeoliteConverter * LHVHazardous * Xincele) / 3.6


# 4. Data for export

In [ ]:
# Useful thermal energy produced by the RICE over the reference study period [kWh]
ThermalEnergyProduced_RICE = (
    np.asarray(ThermalEnergyproduced_primary, dtype=float)
    * RSP
)

# Electricity produced by the RICE over the reference study period [kWh_el]
ElectricEnergyProduced_RICE = RICE_W_RSP